# HDAData package documentation

This package serves as a high-dimensional data analysis tool for Python using bioinformatics data, as part of the DAT-M-HIGHDIM lab project.

The package consists of the following main functionalities:
1. [Loading files and data structures](##1.-Loading-files-and-data-structures)
2. Distance functions for dynamic models and heatmaps
3. Clustering algorithms
4. Dimensionality reduction algorithms

# How to install

...to be updated....

In [ ]:
# temporary

from load.loader import Loader
from structure.object import HDAData

# Tutorial

This tutorial shows how to use the HDAData package following the provided sample data in the `/data` folder as a guide.

> Background on the sample data

**Scifate data** 
- Sci-fate is a high-throughput sequencing method designed to track transcriptional dynamics of cells (i.e cells which are newly synthesized within a specific window of time) as compared to traditional methods which only observe a "static snapshot" of the cell at a specific moment
- By labelling new mRNA with the 4sU molecule, "old" and "new" genetic activity in the cell can be tracked over a 10-hour response
- [Link to paper](https://www.nature.com/articles/s41587-020-0480-9)

<br>

| File name | Description | Information | Dataset size (gene x cell) | 
| --------- | ----------- | ------------ | ------------ |
matrix.mtx.gz | Gene expression matrix | Dataset showing the activity levels of all genes across all individual cells | (43750, 7404) |
4sU.Binom.ntr.mtx.gz | New to Total Ratio matrix | The proportion of a gene's total expression that is "new” | (43750, 7404) |
barcodes.tsv.gz | Barcodes + metadata | Unique molecular tags used to identify cells | (7404, 1) |
features.tsv.gz | Features + metadata | Metadata on genes measured | (43750, 5) |
trajectories_sci.csv | Trajectory table | Trajectory of how cells are connected to each other across time points | (949, 7) |


## 1. Loading files and data structures

- [Load files and Create HDA Object](###-Load-files-and-Create-HDA-Object)
- [Inspect object](###-Inspect-object)
- [View matrix](###-View-matrix)
- [Slicing](###-Slicing)
- [Gene gene expression](###-Get-gene-expression)

### Load files and Create HDA Object

In [ ]:
# Option 1: Load with default /data directory and initialize the HDAData Object directly

loader = Loader()
scifate = loader.load_as_hda()

In [ ]:
# Option 2: Load with specific folder path and initialize the HDAData object manually 

loader = Loader(dir = "your_folder_path")
loaded = loader.load_files()

scifate = HDAData(
    X=loaded["gene_expression_matrix"],
    obs=loaded["barcodes_data"],
    var=loaded["features_data"],
    layers={"ntr": loaded["new_to_total_ratio_matrix"]},
    trajectories=loaded["trajectory_table"]
)

### Inspect object

In [ ]:
# Printing the HDAData object gives you an overview of its elements.

print(scifate)

HDAData object with:
  n_vars (genes): 43750
  n_obs (cells): 7404
  X shape: (43750, 7404)
  layers: ['ntr']
  trajectories: Yes


In [ ]:
# Inspect the Barcodes data (obs).

scifate.obs.head()

,name,cell,time,barcode
cell_name,,,,
A549.4h.AAAAACTCTCTCAA,A549.4h.AAAAACTCTCTCAA,A549,4h,AAAAACTCTCTCAA
A549.6h.AAAAAAGAAGTTCA,A549.6h.AAAAAAGAAGTTCA,A549,6h,AAAAAAGAAGTTCA
A549.8h.AAAAACGGAGAATA,A549.8h.AAAAACGGAGAATA,A549,8h,AAAAACGGAGAATA
A549.4h.AAAACGATCCTGGA,A549.4h.AAAACGATCCTGGA,A549,4h,AAAACGATCCTGGA
A549.6h.AAAAACTGAATTCC,A549.6h.AAAAACTGAATTCC,A549,6h,AAAAACTGAATTCC


In [ ]:
# Inspect the Features data (var).

scifate.var.head()

,gene_id,gene_symbol,feature_type,annotation,length
gene_symbol,,,,,
MTOR-AS1,ENSG00000225602,MTOR-AS1,Gene Expression,Exonic (h.ens90),705
ANGPTL7,ENSG00000171819,ANGPTL7,Gene Expression,Exonic (h.ens90),2238
RPL39P6,ENSG00000238173,RPL39P6,Gene Expression,Exonic (h.ens90),150
SNORA59A,ENSG00000239149,SNORA59A,Gene Expression,Exonic (h.ens90),152
AADACL4,ENSG00000204518,AADACL4,Gene Expression,Exonic (h.ens90),1575


In [ ]:
# Inspect the Trajectories table.

scifate.trajectories.head()

,0h,2h,4h,6h,8h,10h
1,A549.0h.CCGGGCGCTCGACG,A549.2h.GCTGGCGGTTATTG,A549.4h.CTACTGGTCCAGTC,A549.6h.GACAATGCTCGCAA,A549.8h.AAAAAACTACCGAC,A549.10h.AATGCAACCGGATC
2,A549.0h.CGTCCTTAGTTCAA,A549.2h.AGCTACTCCGCCAA,A549.4h.GTTACTCTTAGCGG,A549.6h.AGTCTCGTAGGCTT,A549.8h.AAAAACGGAGAATA,A549.10h.CTCCTGGATCAGGC
3,A549.0h.GGGATAGCTGCTAC,A549.2h.CAGGCGATCATTAC,A549.4h.CAAAGACGAGGTTG,A549.6h.CCTAGATCAGTCAT,A549.8h.AAAATAGAATAGCC,A549.10h.GGCTGACCTGGAAT
4,A549.0h.CGACATAATCTCGC,A549.2h.GCTAACTGGTAGAT,A549.4h.GTTCTCTCTCATCC,A549.6h.GACTGCTCATATGC,A549.8h.AAAATTCTACGCCA,A549.10h.CCGCCTCGGTCAAT
5,A549.0h.CTGTAGGTCTATGG,A549.2h.ATCTATCTAGGTTC,A549.4h.GCGGATCCTACCTC,A549.6h.GCGAGGCCTCCATT,A549.8h.AAACAACTACCGAC,A549.10h.ACCTGGTCCAGGAG


### View matrix

Display matrix (e.g X or ntr) in different formats by first n rows or columns. By default, it displays the gene expression matrix (X). 

Formats supported:
- COO Coordinate List format format (row, column, value) 
    - returns None if it only contains zero entries
- Dense format -> numeric matrix in numpy arrays
- Labelled view -> labelled gene names as rows and cell names as columns in pandas DataFrame

In [ ]:
# Inspect the first 5 rows and columns of the gene expression matrix (X)

# Option 1: COO
print(f'First 5 rows and columns of X in COO format: ')
print(scifate.show(n_rows=5, n_cols=5, mode="coo"))
print(f'-'*20)

# Option 2: Dense
print(f'\nFirst 5 rows and columns of X in Dense format: ')
print(scifate.show(n_rows=5, n_cols=5, mode="dense"))
print(f'-'*20)

# Option 3: Labelled 
print(f'\nFirst 5 rows and columns of X in Label format: ')
df_X = scifate.show(n_rows=5, n_cols=5, mode="label")
df_X


First 5 rows and columns of X in COO format: 
None
--------------------

First 5 rows and columns of X in Dense format: 
[[0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]]
--------------------

First 5 rows and columns of X in Label format: 


cell_name,A549.4h.AAAAACTCTCTCAA,A549.6h.AAAAAAGAAGTTCA,A549.8h.AAAAACGGAGAATA,A549.4h.AAAACGATCCTGGA,A549.6h.AAAAACTGAATTCC
gene_symbol,,,,,
MTOR-AS1,0,0,0,0,0
ANGPTL7,0,0,0,0,0
RPL39P6,0,0,0,0,0
SNORA59A,0,0,0,0,0
AADACL4,0,0,0,0,0


### Slicing

Get subset of HDAData using gene or cell names or index.

In [ ]:
# Example indexing patters:

scifate['Gene1', 'Cell1']                      # Single gene and cell
scifate[['Gene1', 'Gene2'], :]                 # Multiple genes, all cells
scifate[:, ['Cell1', 'Cell2']]                 # All genes, specific cells
scifate['Gene1':'Gene10', 'Cell1':'Cell5']     # Slice by name
scifate[:5, ['Cell1', 'Cell2']]                # Slice by gene index and cell names
scifate[['Gene1', 'Gene2'], :5]                # Slice by gene name and cell index
scifate[10:20, 30:40]                          # Integer indexing

In [ ]:
# Example 1: Get cell using cell name
cell1 = scifate[:, "A549.10h.GAGCTGAGACTCTA"]
cell1.show(n_rows=5, mode="label")

cell_name,A549.10h.GAGCTGAGACTCTA
gene_symbol,
MTOR-AS1,1
ANGPTL7,0
RPL39P6,0
SNORA59A,0
AADACL4,0


In [ ]:
# Example 2: Get gene using gene name
gene1 = scifate["MTOR-AS1", :]
gene1.show(n_cols=5, mode="label")

cell_name,A549.4h.AAAAACTCTCTCAA,A549.6h.AAAAAAGAAGTTCA,A549.8h.AAAAACGGAGAATA,A549.4h.AAAACGATCCTGGA,A549.6h.AAAAACTGAATTCC
gene_symbol,,,,,
MTOR-AS1,0,0,0,0,0


In [ ]:
# Example 3: Subset 10 rows and 10 columns from X
subset = scifate[10:20, 30:40]
subset.show(n_rows=5, n_cols=5, mode="label")

cell_name,A549.4h.AAAATCCGTCGCCA,A549.0h.AAAAACGCGTATCG,A549.2h.AAAACAGCAGTATC,A549.0h.AAAACATTATTACC,A549.10h.AAAACGTATTGAGA
gene_symbol,,,,,
PRAMEF12,0,0,0,0,0
PRAMEF20,0,0,0,0,0
AL354712.1,0,0,0,0,0
RPL7P11,0,0,0,0,0
AL357552.1,0,0,0,0,0


### Get gene expression

Query expression value directly for specific gene and cell.

In [ ]:
# Example 1: get gene expression using gene name and cell name
print(scifate.get_expression("MTOR-AS1", "A549.10h.GAGCTGAGACTCTA"))

# Example 2: get gene expression using gene index and cell index
print(scifate.get_expression(0, 0))

1
